> Mengunduh dataset publik Figshare ke Drive pribadi. **Jangan membagikan tautan folder hasil unduhan.**


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q requests tqdm

In [ ]:
from __future__ import annotations

import argparse
import hashlib
import sys
import zipfile
from pathlib import Path

import requests
from tqdm.auto import tqdm

DEFAULT_OUT = Path("/content/drive/MyDrive/kp2_figshare")


def ensure_drive_mounted() -> None:
    drive_marker = Path("/content/drive/MyDrive")
    if drive_marker.is_dir():
        return
    try:
        from google.colab import drive  # type: ignore
    except ImportError:
        print("[peringatan] Bukan lingkungan Colab atau modul google.colab tidak ada; "
              "lewati mount Drive otomatis.")
        return
    print("Google Drive belum ter-mount, mounting sekarang...")
    drive.mount("/content/drive")

ARTICLE_ID = 14984667
FIGSHARE_API = f"https://api.figshare.com/v2/articles/{ARTICLE_ID}"


def get_file_list() -> list[dict]:
    r = requests.get(FIGSHARE_API, timeout=30)
    r.raise_for_status()
    files = r.json().get("files", [])
    if not files:
        raise RuntimeError("Tidak ada daftar berkas dari Figshare API.")
    return files


def _md5(path: Path, chunk: int = 1 << 20) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        while blk := f.read(chunk):
            h.update(blk)
    return h.hexdigest()


def download_file(url: str, dest: Path, expected_md5: str | None = None) -> Path:
    if dest.exists() and expected_md5 and _md5(dest) == expected_md5:
        print(f"[skip] {dest.name} sudah ada dan checksum cocok")
        return dest

    dest.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(dest, "wb") as f, tqdm(
            total=total, unit="B", unit_scale=True, desc=dest.name
        ) as pbar:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                pbar.update(len(chunk))

    if expected_md5:
        actual = _md5(dest)
        if actual != expected_md5:
            raise RuntimeError(
                f"Checksum tidak cocok untuk {dest.name}: "
                f"diharapkan {expected_md5}, didapat {actual}"
            )
    return dest


def extract_if_archive(path: Path, out_dir: Path) -> None:
    if path.suffix.lower() == ".zip":
        print(f"[extract] {path.name} -> {out_dir}")
        with zipfile.ZipFile(path) as zf:
            zf.extractall(out_dir)


def main(argv=None) -> None:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--out", type=Path, default=DEFAULT_OUT,
                     help="folder tujuan (default: Google Drive, /content/drive/MyDrive/kp2_figshare). "
                          "Pakai /content/figshare_kp2 kalau hanya butuh penyimpanan sementara.")
    ap.add_argument("--keep-archive", action="store_true")
    # argparse aman dipakai di Colab selama tidak ada argumen aneh dari kernel
    args, _unknown = ap.parse_known_args(argv)

    if str(args.out).startswith("/content/drive"):
        ensure_drive_mounted()

    args.out.mkdir(parents=True, exist_ok=True)
    raw_dir = args.out / "_unduhan_mentah"

    print(f"Mengambil daftar berkas dari Figshare (article {ARTICLE_ID})...")
    files = get_file_list()
    print(f"Ditemukan {len(files)} berkas:")
    for f in files:
        print(f"  - {f['name']} ({f['size'] / (1024 * 1024):.1f} MB)")

    for f in files:
        dest = raw_dir / f["name"]
        download_file(f["download_url"], dest, expected_md5=f.get("supplied_md5"))
        extract_if_archive(dest, args.out)
        if dest.suffix.lower() == ".zip" and not args.keep_archive:
            dest.unlink()

    print(f"\nSelesai. Data ada di: {args.out.resolve()}")
    print("Struktur folder (2 level):")
    for p in sorted(args.out.rglob("*"))[:40]:
        rel = p.relative_to(args.out)
        if len(rel.parts) <= 2:
            print(f"  {rel}")


if __name__ == "__main__":
    # Colab menyuntikkan argumen kernel sendiri ke sys.argv; abaikan yang tak dikenal.
    main(sys.argv[1:])